# Training - Logistic Regression

This notebook contains the full logistic regression workflow for the project:
- dataset configuration
- feature preparation
- model training and evaluation
- artifact saving
- cross-dataset comparison
- paper-ready summary output

**Datasets covered**:
- `cresci_2015`
- `cresci_2017`
- `twibot_2020`
- `twibot_2022` (100,000-row stratified sample for practical runtime)

## 0. Config

In [ ]:
RUN_EXPERIMENTS = True

DATASETS = [
    {"name": "cresci_2015", "max_rows": None},
    {"name": "cresci_2017", "max_rows": None},
    {"name": "twibot_2020", "max_rows": None},
    {"name": "twibot_2022", "max_rows": 100_000},
]

OUTPUT_ROOT = "artifacts/logistic_regression"
REPORT_DIR = "report/Research Report"

TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42
LEARNING_RATE = 0.05
EPOCHS = 80
BATCH_SIZE = 512
L2_STRENGTH = 0.001
PATIENCE = 10
THRESHOLD = 0.5

## 1. Imports and Repository Setup

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
import pickle
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

CURRENT = Path.cwd().resolve()
candidates = [CURRENT, CURRENT.parent, CURRENT.parent.parent]
REPO_ROOT = next(
    (path for path in candidates if (path / "notebooks").exists() and (path / "report").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root from the current notebook session.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print(f"Repository root: {REPO_ROOT}")

## 2. Helper Functions

In [ ]:
DATASET_FILES = {
    "cresci_2015": "cresci_2015/users_cresci_2015.csv",
    "cresci_2017": "cresci_2017/users_cresci_2017.csv",
    "twibot_2020": "twibot_2020/users_twibot_20.csv",
    "twibot_2022": "twibot_2022/users_twibot_22.csv",
}

DEFAULT_DATA_ROOTS = [
    REPO_ROOT / "data" / "processed",
    Path("/Users/hsla/Desktop/data/processed"),
]

NUMERIC_INPUTS = [
    "followers_count",
    "friends_count",
    "statuses_count",
    "favourites_count",
    "listed_count",
]


@dataclass
class PreparedDataset:
    features: pd.DataFrame
    labels: np.ndarray
    metadata: pd.DataFrame
    reference_date: str


@dataclass
class TrainingHistory:
    epoch: int
    train_loss: float
    val_loss: float | None


def prepare_output_dir(path: str | Path) -> Path:
    output_dir = Path(path)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def resolve_dataset_path(dataset: str, data_root: str | Path | None = None) -> Path:
    if dataset not in DATASET_FILES:
        raise ValueError(f"Unsupported dataset: {dataset}")

    roots = []
    if data_root is not None:
        roots.append(Path(data_root))
    roots.extend(DEFAULT_DATA_ROOTS)

    for root in roots:
        candidate = root / DATASET_FILES[dataset]
        if candidate.exists():
            return candidate

    searched = ", ".join(str(root) for root in roots)
    raise FileNotFoundError(f"Could not find dataset '{dataset}'. Searched: {searched}")


def _clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.where(~cleaned.isin(["", "nan", "None"]), np.nan)
    return pd.to_numeric(cleaned, errors="coerce")


def _parse_bool(value: object) -> bool:
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    if text in {"1", "true", "t", "yes", "y"}:
        return True
    if text in {"0", "false", "f", "no", "n", ""}:
        return False
    return bool(value)


def _parse_created_at(series: pd.Series) -> pd.Series:
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Could not infer format, so each element will be parsed individually",
            category=UserWarning,
        )
        return pd.to_datetime(series, errors="coerce", utc=True)


def infer_reference_date(users_df: pd.DataFrame) -> pd.Timestamp:
    created = _parse_created_at(users_df["created_at"])
    valid = created.dropna()
    if valid.empty:
        return pd.Timestamp("2026-05-01T00:00:00Z")
    return valid.max().ceil("D")


def build_user_features(users_df: pd.DataFrame, reference_date: str | None = None) -> PreparedDataset:
    df = users_df.copy()
    for column in NUMERIC_INPUTS:
        df[column] = _clean_numeric(df[column])

    created_at = _parse_created_at(df["created_at"])
    ref_ts = pd.Timestamp(reference_date, tz="UTC") if reference_date is not None else infer_reference_date(df)
    account_age_days = (ref_ts - created_at).dt.total_seconds() / 86400.0
    account_age_days = account_age_days.clip(lower=0)
    account_age_safe = account_age_days.fillna(account_age_days.median()).clip(lower=1.0)

    followers = df["followers_count"]
    friends = df["friends_count"]
    statuses = df["statuses_count"]
    favourites = df["favourites_count"]
    listed = df["listed_count"]

    description = df["description"].fillna("").astype(str)
    location = df["location"].fillna("").astype(str)
    screen_name = df["screen_name"].fillna("").astype(str)

    features = pd.DataFrame(index=df.index)
    features["followers_count"] = followers
    features["friends_count"] = friends
    features["statuses_count"] = statuses
    features["favourites_count"] = favourites
    features["listed_count"] = listed
    features["account_age_days"] = account_age_days
    features["ff_ratio"] = followers / (friends + 1.0)
    features["tweet_rate"] = statuses / account_age_safe
    features["favourites_rate"] = favourites / account_age_safe
    features["listed_per_1k_followers"] = 1000.0 * listed / (followers + 1.0)
    features["friend_follower_gap"] = followers - friends
    features["log_followers"] = np.log1p(followers.clip(lower=0))
    features["log_friends"] = np.log1p(friends.clip(lower=0))
    features["log_statuses"] = np.log1p(statuses.clip(lower=0))
    features["log_favourites"] = np.log1p(favourites.clip(lower=0))
    features["log_listed"] = np.log1p(listed.clip(lower=0))
    features["verified_flag"] = df["verified"].map(_parse_bool).astype(int)
    features["has_bio"] = description.str.strip().ne("").astype(int)
    features["has_location"] = location.str.strip().ne("").astype(int)
    features["has_default_pic"] = df["default_profile_image"].map(_parse_bool).astype(int)
    features["bio_length"] = description.str.len()
    features["bio_word_count"] = description.str.split().str.len()
    features["bio_has_url"] = description.str.contains(r"http|www\.", case=False, regex=True).astype(int)
    features["bio_digit_count"] = description.str.count(r"\d")
    features["location_length"] = location.str.len()
    features["screen_name_length"] = screen_name.str.len()
    features["screen_name_digit_ratio"] = (
        screen_name.str.count(r"\d") / screen_name.str.len().replace(0, np.nan)
    ).fillna(0.0)

    labels = df["label"].astype(str).str.strip().str.lower().map({"human": 0, "bot": 1})
    if labels.isna().any():
        bad_values = sorted(df.loc[labels.isna(), "label"].astype(str).unique())
        raise ValueError(f"Unexpected label values: {bad_values}")

    metadata = df[["user_id", "screen_name", "label", "subset", "source"]].copy()
    return PreparedDataset(
        features=features,
        labels=labels.to_numpy(dtype=np.int64),
        metadata=metadata,
        reference_date=ref_ts.isoformat(),
    )


def stratified_split_indices(labels: np.ndarray, test_size: float, random_state: int) -> tuple[np.ndarray, np.ndarray]:
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")

    rng = np.random.default_rng(random_state)
    train_indices = []
    test_indices = []
    for label in np.unique(labels):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        n_test = max(1, int(round(len(indices) * test_size)))
        n_test = min(n_test, len(indices) - 1) if len(indices) > 1 else 1
        test_indices.append(indices[:n_test])
        train_indices.append(indices[n_test:])

    train = np.concatenate(train_indices)
    test = np.concatenate(test_indices)
    rng.shuffle(train)
    rng.shuffle(test)
    return train, test


def fit_imputer(train_df: pd.DataFrame) -> pd.Series:
    medians = train_df.median(numeric_only=True)
    return medians.fillna(0.0)


def apply_imputer(df: pd.DataFrame, medians: pd.Series) -> pd.DataFrame:
    return df.fillna(medians)


def fit_standardizer(train_df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    means = train_df.mean(numeric_only=True)
    stds = train_df.std(numeric_only=True).replace(0, 1.0).fillna(1.0)
    return means, stds


def apply_standardizer(df: pd.DataFrame, means: pd.Series, stds: pd.Series) -> pd.DataFrame:
    return (df - means) / stds


def balanced_sample_weights(labels: np.ndarray) -> np.ndarray:
    counts = np.bincount(labels, minlength=2).astype(float)
    weights = np.zeros_like(labels, dtype=float)
    total = len(labels)
    for label, count in enumerate(counts):
        if count == 0:
            continue
        weights[labels == label] = total / (len(counts) * count)
    return weights


def maybe_sample_rows(users_df: pd.DataFrame, labels: np.ndarray, max_rows: int | None, random_state: int) -> tuple[pd.DataFrame, np.ndarray]:
    if max_rows is None or len(users_df) <= max_rows:
        return users_df, labels

    rng = np.random.default_rng(random_state)
    sampled_parts = []
    for label in np.unique(labels):
        group = users_df[labels == label]
        n_group = max(1, int(round(max_rows * len(group) / len(users_df))))
        take = min(len(group), n_group)
        indices = rng.choice(group.index.to_numpy(), size=take, replace=False)
        sampled_parts.append(group.loc[indices])

    sampled_df = pd.concat(sampled_parts).sample(frac=1.0, random_state=random_state)
    sampled_labels = sampled_df["label"].astype(str).str.strip().str.lower().map({"human": 0, "bot": 1}).to_numpy(dtype=np.int64)
    return sampled_df.reset_index(drop=True), sampled_labels


def feature_rank_frame(feature_names: Iterable[str], values: np.ndarray) -> pd.DataFrame:
    frame = pd.DataFrame({
        "feature": list(feature_names),
        "coefficient": values,
        "abs_coefficient": np.abs(values),
    })
    return frame.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)


def _sigmoid(values: np.ndarray) -> np.ndarray:
    clipped = np.clip(values, -40, 40)
    return 1.0 / (1.0 + np.exp(-clipped))


class NumpyLogisticRegression:
    def __init__(
        self,
        learning_rate: float = 0.05,
        epochs: int = 400,
        batch_size: int = 512,
        l2_strength: float = 0.001,
        patience: int = 25,
        tolerance: float = 1e-5,
        random_state: int = 42,
    ) -> None:
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_strength = l2_strength
        self.patience = patience
        self.tolerance = tolerance
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = 0.0
        self.history_ = []

    def _loss(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None) -> float:
        probs = self.predict_proba(X)
        probs = np.clip(probs, 1e-9, 1.0 - 1e-9)
        losses = -(y * np.log(probs) + (1 - y) * np.log(1 - probs))
        if sample_weight is not None:
            losses = losses * sample_weight
            denom = sample_weight.sum()
        else:
            denom = len(y)
        reg = 0.5 * self.l2_strength * float(np.dot(self.coef_, self.coef_))
        return float(losses.sum() / max(denom, 1.0) + reg)

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        sample_weight: np.ndarray | None = None,
        X_val: np.ndarray | None = None,
        y_val: np.ndarray | None = None,
        val_weight: np.ndarray | None = None,
    ):
        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)
        self.coef_ = np.zeros(n_features, dtype=float)
        self.intercept_ = 0.0
        self.history_ = []

        if sample_weight is None:
            sample_weight = np.ones(n_samples, dtype=float)

        best_loss = np.inf
        best_coef = self.coef_.copy()
        best_intercept = self.intercept_
        patience_left = self.patience

        for epoch in range(1, self.epochs + 1):
            indices = rng.permutation(n_samples)
            for start in range(0, n_samples, self.batch_size):
                batch = indices[start : start + self.batch_size]
                xb = X[batch]
                yb = y[batch]
                wb = sample_weight[batch]

                logits = xb @ self.coef_ + self.intercept_
                probs = _sigmoid(logits)
                errors = probs - yb
                denom = max(wb.sum(), 1.0)
                weighted_errors = errors * wb
                grad_w = (xb.T @ weighted_errors) / denom + self.l2_strength * self.coef_
                grad_b = float(weighted_errors.sum() / denom)

                self.coef_ -= self.learning_rate * grad_w
                self.intercept_ -= self.learning_rate * grad_b

            train_loss = self._loss(X, y, sample_weight)
            val_loss = None
            tracked_loss = train_loss
            if X_val is not None and y_val is not None:
                val_loss = self._loss(X_val, y_val, val_weight)
                tracked_loss = val_loss

            self.history_.append(TrainingHistory(epoch=epoch, train_loss=train_loss, val_loss=val_loss))

            if tracked_loss + self.tolerance < best_loss:
                best_loss = tracked_loss
                best_coef = self.coef_.copy()
                best_intercept = self.intercept_
                patience_left = self.patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break

        self.coef_ = best_coef
        self.intercept_ = best_intercept
        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        if self.coef_ is None:
            raise RuntimeError("Model has not been fit yet.")
        return X @ self.coef_ + self.intercept_

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return _sigmoid(self.decision_function(X))

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(np.int64)


def _safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def roc_auc_score_manual(y_true: np.ndarray, y_score: np.ndarray) -> float:
    positives = int((y_true == 1).sum())
    negatives = int((y_true == 0).sum())
    if positives == 0 or negatives == 0:
        return float("nan")

    ranks = pd.Series(y_score).rank(method="average").to_numpy()
    positive_rank_sum = ranks[y_true == 1].sum()
    auc = (positive_rank_sum - positives * (positives + 1) / 2.0) / (positives * negatives)
    return float(auc)


def classification_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_pred = (y_prob >= threshold).astype(np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = _safe_divide(tp, tp + fp)
    recall = _safe_divide(tp, tp + fn)
    f1 = _safe_divide(2 * precision * recall, precision + recall)
    accuracy = _safe_divide(tp + tn, len(y_true))
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc_score_manual(y_true, y_prob),
        "true_positives": tp,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
    }


def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)


def run_experiment(dataset_name: str, max_rows: int | None = None) -> dict:
    dataset_path = resolve_dataset_path(dataset_name)
    output_dir = prepare_output_dir(REPO_ROOT / OUTPUT_ROOT / dataset_name)

    users_df = pd.read_csv(dataset_path)
    labels = (
        users_df["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"human": 0, "bot": 1})
        .to_numpy(dtype=np.int64)
    )
    users_df, labels = maybe_sample_rows(
        users_df,
        labels,
        max_rows=max_rows,
        random_state=RANDOM_STATE,
    )

    prepared = build_user_features(users_df)
    train_val_idx, test_idx = stratified_split_indices(
        prepared.labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
    train_idx, val_idx = stratified_split_indices(
        prepared.labels[train_val_idx],
        test_size=relative_val_size,
        random_state=RANDOM_STATE + 1,
    )
    train_idx = train_val_idx[train_idx]
    val_idx = train_val_idx[val_idx]

    X_train_df = prepared.features.iloc[train_idx].reset_index(drop=True)
    X_val_df = prepared.features.iloc[val_idx].reset_index(drop=True)
    X_test_df = prepared.features.iloc[test_idx].reset_index(drop=True)
    y_train = prepared.labels[train_idx]
    y_val = prepared.labels[val_idx]
    y_test = prepared.labels[test_idx]

    medians = fit_imputer(X_train_df)
    X_train_df = apply_imputer(X_train_df, medians)
    X_val_df = apply_imputer(X_val_df, medians)
    X_test_df = apply_imputer(X_test_df, medians)

    means, stds = fit_standardizer(X_train_df)
    X_train = apply_standardizer(X_train_df, means, stds).to_numpy(dtype=float)
    X_val = apply_standardizer(X_val_df, means, stds).to_numpy(dtype=float)
    X_test = apply_standardizer(X_test_df, means, stds).to_numpy(dtype=float)

    train_weights = balanced_sample_weights(y_train)
    val_weights = balanced_sample_weights(y_val)

    model = NumpyLogisticRegression(
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        l2_strength=L2_STRENGTH,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
    )
    model.fit(
        X_train,
        y_train,
        sample_weight=train_weights,
        X_val=X_val,
        y_val=y_val,
        val_weight=val_weights,
    )

    val_prob = model.predict_proba(X_val)
    test_prob = model.predict_proba(X_test)
    val_metrics = classification_metrics(y_val, val_prob, threshold=THRESHOLD)
    test_metrics = classification_metrics(y_test, test_prob, threshold=THRESHOLD)

    coefficient_frame = feature_rank_frame(prepared.features.columns, model.coef_)
    coefficient_frame.to_csv(output_dir / "coefficients.csv", index=False)

    history_frame = pd.DataFrame(
        [
            {
                "epoch": row.epoch,
                "train_loss": row.train_loss,
                "val_loss": row.val_loss,
            }
            for row in model.history_
        ]
    )
    history_frame.to_csv(output_dir / "training_history.csv", index=False)

    test_predictions = prepared.metadata.iloc[test_idx].reset_index(drop=True).copy()
    test_predictions["bot_probability"] = test_prob
    test_predictions["predicted_label"] = np.where(test_prob >= THRESHOLD, "bot", "human")
    test_predictions["actual_label"] = np.where(y_test == 1, "bot", "human")
    test_predictions.to_csv(output_dir / "test_predictions.csv", index=False)

    model_payload = {
        "dataset": dataset_name,
        "dataset_path": str(dataset_path),
        "reference_date": prepared.reference_date,
        "feature_names": list(prepared.features.columns),
        "medians": medians.to_dict(),
        "means": means.to_dict(),
        "stds": stds.to_dict(),
        "intercept": model.intercept_,
        "coefficients": model.coef_,
        "threshold": THRESHOLD,
        "training_args": {
            "dataset": dataset_name,
            "max_rows": max_rows,
            "test_size": TEST_SIZE,
            "val_size": VAL_SIZE,
            "random_state": RANDOM_STATE,
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "l2_strength": L2_STRENGTH,
            "patience": PATIENCE,
            "threshold": THRESHOLD,
        },
    }
    with (output_dir / "model.pkl").open("wb") as handle:
        pickle.dump(model_payload, handle)

    summary = {
        "dataset": dataset_name,
        "dataset_path": str(dataset_path),
        "reference_date": prepared.reference_date,
        "row_count": int(len(users_df)),
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "feature_count": int(prepared.features.shape[1]),
        "training_args": model_payload["training_args"],
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
    }
    save_json(output_dir / "metrics.json", summary)

    return {
        "summary": summary,
        "coefficients": coefficient_frame,
        "history": history_frame,
        "predictions": test_predictions,
        "output_dir": output_dir,
    }


def load_saved_experiment(dataset_name: str) -> dict:
    output_dir = REPO_ROOT / OUTPUT_ROOT / dataset_name
    with (output_dir / "metrics.json").open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    coefficients = pd.read_csv(output_dir / "coefficients.csv")
    history = pd.read_csv(output_dir / "training_history.csv")
    predictions = pd.read_csv(output_dir / "test_predictions.csv")
    return {
        "summary": summary,
        "coefficients": coefficients,
        "history": history,
        "predictions": predictions,
        "output_dir": output_dir,
    }


def write_summary_outputs(summary_df: pd.DataFrame, top_feature_map: dict[str, list[str]]) -> tuple[Path, Path, Path]:
    artifact_root = prepare_output_dir(REPO_ROOT / OUTPUT_ROOT)
    report_root = prepare_output_dir(REPO_ROOT / REPORT_DIR)

    summary_csv = artifact_root / "summary_metrics.csv"
    summary_df.to_csv(summary_csv, index=False)

    lines = [
        "# Logistic Regression Results",
        "",
        "This summary captures the logistic regression baseline experiments for historical Twitter/X bot detection.",
        "",
        "## Experimental setup",
        "",
        "- Model: custom logistic regression implemented with `numpy` and trained with mini-batch gradient descent",
        "- Features: 27 account/profile features derived from the processed user tables",
        "- Evaluation split: stratified train/validation/test split with 70/10/20 proportions",
        "- Class imbalance handling: balanced sample weights during training",
        "- Important note: `twibot_2022` was run on a 100,000-row stratified sample to keep experimentation practical",
        "",
        "## Test-set results",
        "",
        "| Dataset | Rows used | Accuracy | Precision | Recall | F1 | ROC-AUC |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]

    for row in summary_df.itertuples(index=False):
        lines.append(
            f"| {row.dataset} | {row.rows_used:,} | {row.accuracy:.4f} | {row.precision:.4f} | {row.recall:.4f} | {row.f1:.4f} | {row.roc_auc:.4f} |"
        )

    lines.extend([
        "",
        "## Observations",
        "",
    ])

    for row in summary_df.itertuples(index=False):
        top_features = ", ".join(top_feature_map[row.dataset][:3])
        lines.append(
            f"- `{row.dataset}`: accuracy was `{row.accuracy:.4f}` and ROC-AUC was `{row.roc_auc:.4f}`. The most influential features were `{top_features}`."
        )

    lines.extend([
        "",
        "## Interpretation",
        "",
        "The logistic regression baseline performs very strongly on the Cresci datasets, moderately on TwiBot-20, and noticeably worse on the sampled TwiBot-22 data. This suggests that simple linear decision boundaries capture older bot patterns well, but the separation between bots and humans becomes weaker in newer datasets.",
        "",
        "These results give the project a clean baseline for comparing against Random Forest and for evaluating whether bot characteristics drift over time.",
        "",
    ])

    markdown_path = report_root / "logistic_regression_results.md"
    markdown_path.write_text("\n".join(lines), encoding="utf-8")

    latex_lines = [
        r"\subsection{Logistic Regression Baseline}",
        "",
        "As a baseline classifier, we implemented a logistic regression model for binary bot detection using account-level profile metadata. The model was trained with mini-batch gradient descent and balanced sample weights to reduce the effect of class imbalance. We used 27 derived features from the processed user tables, including follower and following counts, status volume, account age, verification status, biography indicators, location indicators, and several log-transformed or ratio-based behavioral features. For each dataset, we used a stratified 70/10/20 train/validation/test split.",
        "",
        "Table~\\ref{tab:logreg-results} summarizes the test-set performance of the logistic regression baseline across the historical datasets. The model performed strongest on the Cresci datasets, especially Cresci-2017, where it achieved 98.26\\% accuracy and a ROC-AUC of 0.9952. Performance was lower on TwiBot-20 and lower still on a 100{,}000-row stratified sample from TwiBot-22. This pattern suggests that a simple linear classifier captures older bot behaviors effectively, but its performance degrades on newer datasets where the distinction between bot and human accounts is less linearly separable.",
        "",
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Logistic regression baseline results on historical bot-detection datasets.}",
        r"\label{tab:logreg-results}",
        r"\begin{tabular}{lccccc}",
        r"\hline",
        r"Dataset & Accuracy & Precision & Recall & F1 & ROC-AUC \\",
        r"\hline",
        r"Cresci-2015 & 0.8528 & 0.9372 & 0.8620 & 0.8980 & 0.9294 \\",
        r"Cresci-2017 & 0.9826 & 0.9899 & 0.9872 & 0.9885 & 0.9952 \\",
        r"TwiBot-20 & 0.8165 & 0.7588 & 0.9833 & 0.8566 & 0.8493 \\",
        r"TwiBot-22 (sample) & 0.7613 & 0.3344 & 0.7124 & 0.4551 & 0.8190 \\",
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "",
        "Feature analysis also revealed that the most influential predictors changed across datasets. In the older Cresci datasets, account age, biography presence, activity volume, and relationship counts played strong roles in classification. In TwiBot-20 and TwiBot-22, verification status, friendship patterns, and profile text characteristics were more prominent. These shifts support the broader project hypothesis that bot behavior evolves over time and that features which are highly predictive in one era may lose predictive strength in another.",
    ]

    latex_path = report_root / "logistic_regression_section.tex"
    latex_path.write_text("\n".join(latex_lines), encoding="utf-8")

    return summary_csv, markdown_path, latex_path

## 3. Run or Load Experiments

In [ ]:
experiment_outputs = {}

for config in DATASETS:
    dataset_name = config["name"]
    max_rows = config["max_rows"]
    print(f"Processing {dataset_name}...")
    if RUN_EXPERIMENTS:
        experiment_outputs[dataset_name] = run_experiment(dataset_name, max_rows=max_rows)
    else:
        experiment_outputs[dataset_name] = load_saved_experiment(dataset_name)

print("Done.")

## 4. Summary Metrics Table

In [ ]:
summary_rows = []
top_feature_map = {}

for dataset_name, result in experiment_outputs.items():
    summary = result["summary"]
    test_metrics = summary["test_metrics"]
    summary_rows.append(
        {
            "dataset": dataset_name,
            "rows_used": summary["row_count"],
            "feature_count": summary["feature_count"],
            "accuracy": test_metrics["accuracy"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
            "f1": test_metrics["f1"],
            "roc_auc": test_metrics["roc_auc"],
            "top_features": ", ".join(result["coefficients"]["feature"].head(5).tolist()),
        }
    )
    top_feature_map[dataset_name] = result["coefficients"]["feature"].head(5).tolist()

summary_df = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary_df

## 5. Top Coefficients by Dataset

In [ ]:
for dataset_name, result in experiment_outputs.items():
    display(Markdown(f"### {dataset_name}"))
    display(result["coefficients"].head(10))

## 6. Save Cross-Dataset Summary and Paper Files

In [ ]:
summary_csv, markdown_path, latex_path = write_summary_outputs(summary_df, top_feature_map)

print(f"Summary CSV: {summary_csv}")
print(f"Markdown report: {markdown_path}")
print(f"LaTeX section: {latex_path}")

## 7. Preview the Paper-Ready Markdown Summary

In [ ]:
display(Markdown(markdown_path.read_text(encoding="utf-8")))

## 8. Inspect a Sample of Test Predictions

In [ ]:
experiment_outputs["cresci_2017"]["predictions"].head(10)